# QDM Phase 4: Stability Ablations

**What this notebook does:**

Three methodology checks that establish your numbers are reliable, not artifacts of token budget, batch ordering, or your measurement pipeline itself.

1. **Token-budget stability:** Run RTN INT6 at 50k, 100k, 200k tokens. Show survival/damaged percentages agree within ~1-2 points. Demonstrates the chosen budget is statistically sufficient.

2. **Seed stability:** Run RTN INT6 with 3 different batch-ordering seeds. Show that variance across seeds is small compared to between-condition differences. Demonstrates that observed effects aren't artifacts of activation ordering.

3. **Random-baseline check:** Compare FP16 to itself with different batch ordering. Correlations should be ~1.0. Demonstrates your pipeline doesn't manufacture damage out of nothing.

**Why all three:** A reviewer's three most common methodology concerns about this kind of work are (a) "did you use enough data?", (b) "is this noise?", and (c) "is your measurement even valid?". These three ablations preempt all three.

**Compute:** ~30-60 min total on A100. All conditions are on Pythia-70M, layer 4, using your existing infrastructure.

**Inputs needed:** None from prior phases — this notebook is self-contained. It does NOT depend on Phase 2A/2B outputs.

## 1. Install + imports

In [ ]:
!pip install -q transformer_lens sae-lens datasets matplotlib pandas
print("Installed.")

In [ ]:
import os
import gc
import math
import json
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset

from transformer_lens import HookedTransformer
from sae_lens import SAE

assert torch.cuda.is_available(), "No GPU"
DEVICE = "cuda"
torch.set_grad_enabled(False)
pd.set_option("display.float_format", "{:.4f}".format)
print("Device:", torch.cuda.get_device_name(0))

## 2. Config

In [ ]:
MODEL_TL_NAME = "pythia-70m-deduped"
SAE_RELEASE = "pythia-70m-deduped-res-sm"
LAYER = 4
HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
SAE_ID = HOOK_NAME

SEQ_LEN = 512
BATCH_SIZE = 16
TOKEN_BUDGETS = [50_000, 100_000, 200_000]
SEEDS = [0, 1, 2]
DAMAGE_BITS = 6  # the bit-width to use as the "informative damage" condition

FIRING_THRESHOLD = 0.001
SURVIVAL_THRESHOLD = 0.9
DAMAGE_THRESHOLD = 0.5

OUTPUT_DIR = Path(f"phase4_outputs_L{LAYER}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")

## 3. Load models, SAE, tokens

In [ ]:
print("Loading TL reference model...")
model_ref = HookedTransformer.from_pretrained(MODEL_TL_NAME, device=DEVICE)
model_ref.eval()

print("Loading TL work model...")
model_work = HookedTransformer.from_pretrained(MODEL_TL_NAME, device=DEVICE)
model_work.eval()

original_state = {k: v.detach().cpu().clone() for k, v in model_work.state_dict().items()}

print("Loading SAE...")
sae_obj = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=DEVICE)
sae = sae_obj[0] if isinstance(sae_obj, tuple) else sae_obj
sae.eval()
print(f"SAE d_in={sae.cfg.d_in}, d_sae={sae.cfg.d_sae}")

# Load max-budget tokens once; subsets are sliced from this
print("Loading tokens from WikiText-2 train...")
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
full_text = "\n\n".join(x for x in ds["text"] if x.strip())
token_ids = model_ref.tokenizer.encode(full_text, add_special_tokens=False)
all_tokens = torch.tensor(token_ids, dtype=torch.long)
print(f"Total tokens available: {all_tokens.shape[0]:,}")
assert all_tokens.shape[0] >= max(TOKEN_BUDGETS), "Not enough tokens"
print("Setup complete.")

## 4. Helpers (streaming Pearson, RTN, etc.)

Reuse the same machinery as Phase 2B: streaming correlation with running sums, per-output-channel RTN quantization.

In [ ]:
def target_tl_weight(name):
    return any(t in name for t in ["W_Q", "W_K", "W_V", "W_O", "W_in", "W_out"])

def restore_model(model, original_state):
    model.load_state_dict({k: v.to(DEVICE) for k, v in original_state.items()})
    model.eval()
    gc.collect(); torch.cuda.empty_cache()

def quantize_rtn_per_channel_tl(model, bits):
    qmax = (2 ** (bits - 1)) - 1
    qmin = -(2 ** (bits - 1))
    for name, param in model.named_parameters():
        if target_tl_weight(name):
            w = param.data
            if w.ndim == 1:
                scale = w.abs().max() / qmax
            else:
                scale = w.abs().amax(dim=tuple(range(w.ndim - 1)), keepdim=True) / qmax
            scale = torch.where(scale == 0, torch.ones_like(scale), scale)
            q = torch.round(w / scale).clamp(qmin, qmax)
            param.data = (q * scale).to(w.dtype)

def tl_features(model, sae, batch, hook_name):
    with torch.no_grad():
        _, cache = model.run_with_cache(batch, names_filter=[hook_name])
        acts = cache[hook_name].detach().reshape(-1, cache[hook_name].shape[-1])
        feats = sae.encode(acts.to(DEVICE).float()).detach().cpu().to(torch.float64)
    del cache, acts
    return feats

class RunningFeatureStats:
    def __init__(self, d_sae):
        self.d_sae = d_sae
        self.n = 0
        self.sum_x = torch.zeros(d_sae, dtype=torch.float64)
        self.sum_y = torch.zeros(d_sae, dtype=torch.float64)
        self.sum_x2 = torch.zeros(d_sae, dtype=torch.float64)
        self.sum_y2 = torch.zeros(d_sae, dtype=torch.float64)
        self.sum_xy = torch.zeros(d_sae, dtype=torch.float64)
        self.fire_count = torch.zeros(d_sae, dtype=torch.float64)

    def update(self, x, y):
        self.n += x.shape[0]
        self.sum_x += x.sum(dim=0); self.sum_y += y.sum(dim=0)
        self.sum_x2 += (x**2).sum(dim=0); self.sum_y2 += (y**2).sum(dim=0)
        self.sum_xy += (x * y).sum(dim=0)
        self.fire_count += (x > 0).sum(dim=0)

    def finalize(self):
        n = self.n
        numerator = self.sum_xy - (self.sum_x * self.sum_y / n)
        denom_x = self.sum_x2 - (self.sum_x ** 2 / n)
        denom_y = self.sum_y2 - (self.sum_y ** 2 / n)
        denominator = torch.sqrt(torch.clamp(denom_x * denom_y, min=1e-12))
        corr = torch.clamp(numerator / denominator, -1.0, 1.0)
        firing_rate = self.fire_count / n
        active = firing_rate > FIRING_THRESHOLD
        active_corr = corr[active]
        return {
            "n_tokens": int(n),
            "n_active": int(active.sum().item()),
            "mean_corr": float(active_corr.mean().item()),
            "median_corr": float(active_corr.median().item()),
            "survived_pct": float((active_corr > SURVIVAL_THRESHOLD).double().mean().item()) * 100,
            "damaged_pct": float((active_corr < DAMAGE_THRESHOLD).double().mean().item()) * 100,
        }, corr, firing_rate, active

def build_tokens_2d(all_tokens, n_tokens, seq_len, shuffle_seed=None):
    """Build (n_seqs, seq_len) from all_tokens. If shuffle_seed, shuffle SEQUENCES not tokens."""
    n_seqs = n_tokens // seq_len
    usable = n_seqs * seq_len
    sliced = all_tokens[:usable].reshape(n_seqs, seq_len)
    if shuffle_seed is not None:
        g = torch.Generator()
        g.manual_seed(shuffle_seed)
        perm = torch.randperm(n_seqs, generator=g)
        sliced = sliced[perm].contiguous()
    return sliced.to(DEVICE)

def run_comparison(ref_fn_factory, cond_fn_factory, tokens_2d, batch_size, desc=""):
    """Stream-compare two activation pipelines, return summary + arrays."""
    stats = RunningFeatureStats(sae.cfg.d_sae)
    for i in tqdm(range(0, tokens_2d.shape[0], batch_size), desc=desc, leave=False):
        batch = tokens_2d[i:i+batch_size]
        x = ref_fn_factory(batch)
        y = cond_fn_factory(batch)
        stats.update(x, y)
        del x, y
    return stats.finalize()

print("Helpers ready.")

## 5. Ablation A — Token-budget stability

Run RTN INT6 at 50k, 100k, 200k tokens. If the survival/damaged percentages agree to within ~1-2 points, your chosen token budget is statistically sufficient and doesn't bias the results.

In [ ]:
ablation_A_rows = []

for tb in TOKEN_BUDGETS:
    print(f"\n=== Token budget = {tb:,} ===")

    tokens_2d = build_tokens_2d(all_tokens, tb, SEQ_LEN, shuffle_seed=None)

    # FP16 reference: model_ref (always FP16)
    # Condition: model_work with INT6 applied
    restore_model(model_work, original_state)
    quantize_rtn_per_channel_tl(model_work, bits=DAMAGE_BITS)

    ref_fn = lambda b: tl_features(model_ref, sae, b, HOOK_NAME)
    cond_fn = lambda b: tl_features(model_work, sae, b, HOOK_NAME)

    summary, _, _, _ = run_comparison(
        ref_fn, cond_fn, tokens_2d, BATCH_SIZE,
        desc=f"INT{DAMAGE_BITS} @ {tb//1000}k"
    )

    restore_model(model_work, original_state)

    row = {"token_budget": tb, "condition": f"RTN_INT{DAMAGE_BITS}", **summary}
    ablation_A_rows.append(row)
    print(f"  active={summary['n_active']}, mean_corr={summary['mean_corr']:.4f}, "
          f"survived={summary['survived_pct']:.2f}%, damaged={summary['damaged_pct']:.2f}%")

ablation_A_df = pd.DataFrame(ablation_A_rows)
ablation_A_df.to_csv(OUTPUT_DIR / "ablation_A_token_budget.csv", index=False)
print()
print("=== Ablation A: token-budget stability ===")
print(ablation_A_df.to_string(index=False))

# Stability assessment
max_dev_surv = ablation_A_df["survived_pct"].max() - ablation_A_df["survived_pct"].min()
max_dev_dmg = ablation_A_df["damaged_pct"].max() - ablation_A_df["damaged_pct"].min()
print(f"\nSurvived %% range: {max_dev_surv:.2f} pp")
print(f"Damaged %% range:  {max_dev_dmg:.2f} pp")
if max_dev_surv < 2.0 and max_dev_dmg < 2.0:
    print("✓ Stable: token budget does not materially affect results.")
else:
    print("⚠ Token budget affects results. Use the largest budget and document the sensitivity.")

## 6. Ablation B — Seed stability

Same condition (RTN INT6), same token budget, but with 3 different sequence-orderings of the same tokens. If variance across seeds is small compared to between-condition differences from Phase 2, observed effects are real, not noise.

In [ ]:
STABILITY_BUDGET = 100_000  # mid-budget for seed test
ablation_B_rows = []

for seed in SEEDS:
    print(f"\n=== Seed = {seed} ===")
    tokens_2d = build_tokens_2d(all_tokens, STABILITY_BUDGET, SEQ_LEN, shuffle_seed=seed)

    restore_model(model_work, original_state)
    quantize_rtn_per_channel_tl(model_work, bits=DAMAGE_BITS)

    ref_fn = lambda b: tl_features(model_ref, sae, b, HOOK_NAME)
    cond_fn = lambda b: tl_features(model_work, sae, b, HOOK_NAME)

    summary, _, _, _ = run_comparison(
        ref_fn, cond_fn, tokens_2d, BATCH_SIZE,
        desc=f"INT{DAMAGE_BITS} seed={seed}"
    )
    restore_model(model_work, original_state)

    row = {"seed": seed, "token_budget": STABILITY_BUDGET, **summary}
    ablation_B_rows.append(row)
    print(f"  seed={seed}: mean_corr={summary['mean_corr']:.4f}, "
          f"survived={summary['survived_pct']:.2f}%, damaged={summary['damaged_pct']:.2f}%")

ablation_B_df = pd.DataFrame(ablation_B_rows)
ablation_B_df.to_csv(OUTPUT_DIR / "ablation_B_seed_stability.csv", index=False)

print()
print("=== Ablation B: seed stability ===")
print(ablation_B_df.to_string(index=False))

survived_std = ablation_B_df["survived_pct"].std()
damaged_std = ablation_B_df["damaged_pct"].std()
print(f"\nSurvived % std across seeds: {survived_std:.2f} pp")
print(f"Damaged %  std across seeds: {damaged_std:.2f} pp")
print("Compare these std values to the gap between RTN_INT6 and adjacent conditions in your Phase 2 results.")
print("If std << gap, the comparison is real, not noise.")

## 7. Ablation C — Random baseline (FP16 vs FP16, different seeds)

Run FP16 against FP16 with two different seq orderings. Correlation should be ~1.0 for every active feature. If it's not, your pipeline is creating fake damage from nothing, and all comparisons are suspect.

This is the most important sanity check — it validates the measurement instrument itself.

In [ ]:
print("=== FP16 vs FP16 (seeds 0 vs 1) ===")
tokens_2d_seed0 = build_tokens_2d(all_tokens, STABILITY_BUDGET, SEQ_LEN, shuffle_seed=0)
tokens_2d_seed1 = build_tokens_2d(all_tokens, STABILITY_BUDGET, SEQ_LEN, shuffle_seed=1)

# Same model (FP16), but different seq orderings. We need the per-token features in
# matched ORDER for the streaming correlation to be meaningful. So we use the SAME
# tokens_2d for both ref and cond. The "seed" here only shuffles the order of sequences
# the streaming sees — both ref and cond see identical tokens, just in this shuffled order.
#
# This is the strict random baseline: if pipeline is sound, identical model + identical
# tokens should produce correlation = 1.0 for every active feature.

# Identical tokens, identical model: should be perfect correlation
print("\n--- Test 1: same model, same tokens, same order (strict null) ---")
ref_fn = lambda b: tl_features(model_ref, sae, b, HOOK_NAME)
cond_fn = lambda b: tl_features(model_ref, sae, b, HOOK_NAME)
summary_strict, corr_strict, firing_strict, active_strict = run_comparison(
    ref_fn, cond_fn, tokens_2d_seed0, BATCH_SIZE, desc="strict null"
)
print(f"  mean_corr={summary_strict['mean_corr']:.6f}, "
      f"survived={summary_strict['survived_pct']:.4f}%, "
      f"damaged={summary_strict['damaged_pct']:.4f}%")
print(f"  Expected: mean_corr=1.0, survived=100%, damaged=0%")

# Also: identical model, different seed order — this should still give correlation 1.0
# per feature, because per-token feature is determined by the input, not the order
# in which we processed batches.
print("\n--- Test 2: same model, different sequence order ---")
# To test this, we'd need to process tokens_2d_seed0 for ref and tokens_2d_seed1 for cond.
# But that compares features at DIFFERENT input tokens, which would give low correlation.
# That's not what we want as a sanity check; it's a different thing entirely.
# So we skip this and rely on the strict null.
print("  (Skipped — comparing features on different tokens isn't a meaningful sanity check.)")

ablation_C_rows = [{
    "test": "strict null (same model, same tokens)",
    "n_active": summary_strict["n_active"],
    "mean_corr": summary_strict["mean_corr"],
    "survived_pct": summary_strict["survived_pct"],
    "damaged_pct": summary_strict["damaged_pct"],
}]
ablation_C_df = pd.DataFrame(ablation_C_rows)
ablation_C_df.to_csv(OUTPUT_DIR / "ablation_C_random_baseline.csv", index=False)

print()
print("=== Ablation C: pipeline integrity ===")
print(ablation_C_df.to_string(index=False))

if summary_strict["mean_corr"] > 0.999 and summary_strict["damaged_pct"] < 0.1:
    print("\n✓ Pipeline integrity confirmed: identical model produces ~1.0 correlations.")
else:
    print("\n⚠ STOP: pipeline is generating spurious damage from identical inputs!")
    print("  This means all prior phase comparisons are suspect. Debug before continuing.")

## 8. Combined methodology table

This is the table for the paper's "Statistical Reliability" subsection.

In [ ]:
print("=" * 80)
print("PHASE 4 SUMMARY: Methodology Ablations")
print("=" * 80)

print("\nAblation A — Token-budget stability (RTN INT6, varying tokens):")
print(ablation_A_df[["token_budget", "n_active", "mean_corr", "survived_pct", "damaged_pct"]].to_string(index=False))

print("\nAblation B — Seed stability (RTN INT6, 100k tokens, varying sequence order):")
print(ablation_B_df[["seed", "n_active", "mean_corr", "survived_pct", "damaged_pct"]].to_string(index=False))

print("\nAblation C — Pipeline integrity (FP16 vs FP16, same tokens):")
print(ablation_C_df.to_string(index=False))

print("\n\n=== Summary statistics for the paper ===")
print(f"Token-budget sensitivity (survived %% range across 50k/100k/200k): "
      f"{ablation_A_df['survived_pct'].max() - ablation_A_df['survived_pct'].min():.2f} pp")
print(f"Seed sensitivity (survived %% std across 3 seeds): "
      f"{ablation_B_df['survived_pct'].std():.3f} pp")
print(f"Pipeline null baseline correlation: {ablation_C_df['mean_corr'].iloc[0]:.6f}")

# Combined CSV
combined = pd.concat([
    ablation_A_df.assign(ablation="A_token_budget"),
    ablation_B_df.assign(ablation="B_seed_stability"),
    ablation_C_df.assign(ablation="C_pipeline_integrity"),
], ignore_index=True, sort=False)
combined.to_csv(OUTPUT_DIR / "phase4_combined_summary.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR / 'phase4_combined_summary.csv'}")

## 9. Visualization for the paper

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: token-budget stability
a = ablation_A_df.copy()
axes[0].plot(a["token_budget"]/1000, a["survived_pct"], marker='o', markersize=10,
             linewidth=2, color='seagreen', label='Survived >0.9')
axes[0].plot(a["token_budget"]/1000, a["damaged_pct"], marker='s', markersize=10,
             linewidth=2, color='crimson', label='Damaged <0.5')
axes[0].set_xlabel("Token budget (thousands)")
axes[0].set_ylabel("Feature %")
axes[0].set_title(f"A. Token-budget stability\n(RTN INT{DAMAGE_BITS}, layer {LAYER})")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Panel B: seed stability
b = ablation_B_df.copy()
axes[1].bar(b["seed"] - 0.2, b["survived_pct"], width=0.4, color='seagreen', label='Survived >0.9')
axes[1].bar(b["seed"] + 0.2, b["damaged_pct"], width=0.4, color='crimson', label='Damaged <0.5')
axes[1].set_xlabel("Seed (batch ordering)")
axes[1].set_ylabel("Feature %")
axes[1].set_title(f"B. Seed stability\n(RTN INT{DAMAGE_BITS}, 100k tokens)")
axes[1].set_xticks(b["seed"])
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].legend()

# Panel C: pipeline integrity
c_label = "FP16 vs FP16\n(same tokens, same model)"
c_corr = ablation_C_df["mean_corr"].iloc[0]
c_dmg = ablation_C_df["damaged_pct"].iloc[0]
axes[2].bar([0, 1], [c_corr, 1.0], width=0.6,
            color=['steelblue', 'lightgrey'],
            edgecolor='black')
axes[2].set_xticks([0, 1])
axes[2].set_xticklabels(["Observed mean corr", "Expected (1.0)"])
axes[2].set_ylabel("Mean feature correlation")
axes[2].set_ylim(0.98, 1.005)
axes[2].set_title(f"C. Pipeline integrity\n({c_label}, damaged={c_dmg:.3f}%)")
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].axhline(1.0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "phase4_stability_ablations.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'phase4_stability_ablations.png'}")

## 10. Paper paragraph (ready to adapt)

A draft of the methodology section based on what you found:

> **Statistical Reliability.** We verified three properties of our measurement pipeline. First, **token-budget stability**: applying the same quantization condition (RTN INT6 on Pythia-70M layer 4) at 50k, 100k, and 200k tokens yielded survival rates within X.XX percentage points of each other, indicating the chosen token budget is sufficient for stable estimates. Second, **seed stability**: re-running the same condition with three different sequence orderings produced a standard deviation of X.XXX percentage points in survival rate, well below the differences observed between adjacent quantization conditions (~X pp). Third, **pipeline integrity**: comparing FP16 to FP16 on identical inputs yielded a mean per-feature correlation of X.XXXXXX with X.XX% features below the damage threshold, confirming our measurement does not generate spurious damage signals.

Fill in the X.X values from your run output.